<a href="https://colab.research.google.com/github/MayerT1/Prep_GEDI/blob/main/Prep_Interence_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://code.earthengine.google.com/8c292ac4ed9f537ec80641a49fab8e52

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
import os

# Define the base project folder and subdirectories
base_dir = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project'
subdirs = [
    '2024_Imagery_For_Inference'
    'data_32_32_patches_11_4_25',
    'target_data',
    'data',
    'data_NaN_filtered',
    'models',
    "model_animations",
    'scripts',
    'notebooks',
    'config',
    'results',
    'prediction_surface'
]

# Create each subdirectory
for subdir in subdirs:
    path = os.path.join(base_dir, subdir)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inferencedata_32_32_patches_11_4_25
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/target_data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data_NaN_filtered
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/models
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/model_animations
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/scripts
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/notebooks
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/config
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/results
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/prediction_surface


In [7]:
%cd '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference'

/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference


In [8]:
!ls

32x32_patches	     LandsatComposite_2024.tif		 s1Ascending_2024.tif
DEMindices_2024.tif  LandsatIndices_2024.tif		 S2Composite_2024.tif
HLS_2024.tif	     landsatTasseledCapIndices_2024.tif  S2Indices_2024.tif


In [10]:
# !rm -f /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference/32x32_patches/*.tif


In [11]:
# Run this in Colab (assumes drive already mounted)
# Example: from google.colab import drive; drive.mount('/content/drive')

import os
import math
from pathlib import Path
import rasterio
from rasterio.windows import Window
from tqdm import tqdm

# -----------------------
# CONFIG
# -----------------------
INPUT_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference"
OUTPUT_DIR = os.path.join(INPUT_DIR, "32x32_patches")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Files (use the exact filenames you have)
tif_files = [
    "s1Ascending_2024.tif",
    "DEMindices_2024.tif",
    "HLS_2024.tif",
    "LandsatComposite_2024.tif",
    "LandsatIndices_2024.tif",
    "landsatTasseledCapIndices_2024.tif",
    "S2Composite_2024.tif",
    "S2Indices_2024.tif",
]

# Pixel-constant config
PATCH_SIZE = 32             # pixels (constant for all sensors)
OVERLAP_FRACTION = 0.25     # 25% overlap
STRIDE = int(PATCH_SIZE * (1 - OVERLAP_FRACTION))  # = 24

# Safety check
if STRIDE <= 0:
    raise ValueError("Computed stride <= 0. Check PATCH_SIZE and OVERLAP_FRACTION.")

# -----------------------
# FUNCTIONS
# -----------------------
def create_patches_for_tif(tif_path, output_dir=OUTPUT_DIR, patch_size=PATCH_SIZE, stride=STRIDE):
    """
    Create 32x32 pixel patches with 25% overlap for a single GeoTIFF.
    Returns number of patches written.
    """
    tif_path = Path(tif_path)
    if not tif_path.exists():
        raise FileNotFoundError(f"{tif_path} not found")

    base_name = tif_path.stem
    print(f"\nProcessing: {tif_path.name}")
    print(f"  Patch size (pixels): {patch_size} x {patch_size}")
    print(f"  Overlap: {int(OVERLAP_FRACTION*100)}%  -> Stride: {stride} px")

    with rasterio.open(str(tif_path)) as src:
        width, height = src.width, src.height
        print(f"  Source size: width={width} px, height={height} px, bands={src.count}")
        # compute number of positions (rows, cols) in the grid
        n_rows = math.ceil((height - patch_size) / stride) + 1
        n_cols = math.ceil((width - patch_size) / stride) + 1
        expected = n_rows * n_cols
        print(f"  Grid positions: {n_rows} rows x {n_cols} cols -> expected patches = {expected}")

        patch_count = 0
        # iterate grid
        for r in tqdm(range(n_rows), desc="  rows"):
            for c in range(n_cols):
                row_off = r * stride
                col_off = c * stride

                # ensure last patch stays inside image bounds (shift if necessary)
                if row_off + patch_size > height:
                    row_off = height - patch_size
                if col_off + patch_size > width:
                    col_off = width - patch_size

                # Defensive: skip positions that would be negative (tiny images)
                if row_off < 0 or col_off < 0:
                    continue

                window = Window(col_off, row_off, patch_size, patch_size)
                transform = rasterio.windows.transform(window, src.transform)

                out_name = f"{base_name}_{r:04d}_{c:04d}.tif"
                out_path = os.path.join(output_dir, out_name)

                # Write patch preserving dtype, count, crs, transform
                profile = src.profile.copy()
                profile.update({
                    "height": patch_size,
                    "width": patch_size,
                    "transform": transform,
                    "driver": "GTiff",
                    # optional: compression to reduce space (uncomment if desired)
                    # "compress": "DEFLATE",
                })

                with rasterio.open(out_path, "w", **profile) as dst:
                    dst.write(src.read(window=window))

                patch_count += 1

        print(f"  -> Wrote {patch_count} patches for {tif_path.name}")
        return patch_count

# -----------------------
# RUN (single file or all)
# -----------------------
if __name__ == "__main__":
    # Example usage:
    # 1) To run on a single file, set single_file to the filename
    # 2) To run on all files in tif_files, set single_file = None

    single_file = None  # <-- set e.g. "s1Ascending_2024.tif" to process just one file

    total = 0
    per_file_counts = {}

    if single_file:
        path = os.path.join(INPUT_DIR, single_file)
        if os.path.exists(path):
            count = create_patches_for_tif(path)
            per_file_counts[single_file] = count
            total += count
        else:
            print(f"File not found: {single_file}")
    else:
        # process all listed files
        for fname in tif_files:
            path = os.path.join(INPUT_DIR, fname)
            if os.path.exists(path):
                try:
                    count = create_patches_for_tif(path)
                    per_file_counts[fname] = count
                    total += count
                except Exception as e:
                    print(f"Error processing {fname}: {e}")
            else:
                print(f"Missing file (skipped): {fname}")

    # Summary
    print("\n=======================================")
    print(f"TOTAL PATCHES CREATED: {total}")
    print("Patches per file:")
    for k, v in per_file_counts.items():
        print(f"  {k}: {v}")
    print("Output directory:", OUTPUT_DIR)
    print("=======================================")



Processing: s1Ascending_2024.tif
  Patch size (pixels): 32 x 32
  Overlap: 25%  -> Stride: 24 px
  Source size: width=1268 px, height=1256 px, bands=4
  Grid positions: 52 rows x 53 cols -> expected patches = 2756


  rows: 100%|██████████| 52/52 [01:44<00:00,  2.00s/it]


  -> Wrote 2756 patches for s1Ascending_2024.tif

Processing: DEMindices_2024.tif
  Patch size (pixels): 32 x 32
  Overlap: 25%  -> Stride: 24 px
  Source size: width=423 px, height=419 px, bands=2
  Grid positions: 18 rows x 18 cols -> expected patches = 324


  rows: 100%|██████████| 18/18 [00:08<00:00,  2.17it/s]


  -> Wrote 324 patches for DEMindices_2024.tif

Processing: HLS_2024.tif
  Patch size (pixels): 32 x 32
  Overlap: 25%  -> Stride: 24 px
  Source size: width=423 px, height=419 px, bands=7
  Grid positions: 18 rows x 18 cols -> expected patches = 324


...

  -> Wrote 324 patches for landsatTasseledCapIndices_2024.tif

Processing: S2Composite_2024.tif
  Patch size (pixels): 32 x 32
  Overlap: 25%  -> Stride: 24 px
  Source size: width=634 px, height=628 px, bands=6
  Grid positions: 26 rows x 27 cols -> expected patches = 702


  rows: 100%|██████████| 26/26 [00:24<00:00,  1.05it/s]


  -> Wrote 702 patches for S2Composite_2024.tif

Processing: S2Indices_2024.tif
  Patch size (pixels): 32 x 32
  Overlap: 25%  -> Stride: 24 px
  Source size: width=634 px, height=628 px, bands=8
  Grid positions: 26 rows x 27 cols -> expected patches = 702


  rows: 100%|██████████| 26/26 [00:28<00:00,  1.11s/it]

  -> Wrote 702 patches for S2Indices_2024.tif

TOTAL PATCHES CREATED: 5780
Patches per file:
  s1Ascending_2024.tif: 2756
  DEMindices_2024.tif: 324
  HLS_2024.tif: 324
  LandsatComposite_2024.tif: 324
  LandsatIndices_2024.tif: 324
  landsatTasseledCapIndices_2024.tif: 324
  S2Composite_2024.tif: 702
  S2Indices_2024.tif: 702
Output directory: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inference/32x32_patches
